# DreamSpace: train the door & window detector

Trains a small YOLO model on FloorPlanCAD to find single doors, double doors, sliding doors and
windows. Free: it uses Colab's free GPU and your Google Drive. **Nothing here needs a card. If anything
asks for payment, stop.**

**How to run**
1. Menu **Runtime > Change runtime type > T4 GPU**, then Save.
2. Menu **Runtime > Run all**. Colab asks to access your Google Drive: allow it (progress is saved there).
3. Keep the tab open. Expect 2-4 hours.
4. If the session ends, just press **Run all** again: training resumes from the last epoch.
5. When finished, download `DreamSpace_training/dreamspace_door_window.pt` from your Google Drive
   and put it in `cad update folder/training/` in the project.

Dataset licence: FloorPlanCAD is non-commercial (CC BY-NC 4.0). YOLO is AGPL-3.0.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: use Runtime > Change runtime type > T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Install YOLO and connect Google Drive

In [ ]:
!pip -q install ultralytics==8.4.202 huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/DreamSpace_training'
!mkdir -p "$PROJECT"

## 3. Download FloorPlanCAD (454 MB, about 5,300 plan images)

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id='Voxel51/FloorPlanCAD', repo_type='dataset', local_dir='/content/fpc',
                  allow_patterns=['samples.json', 'data/*'], max_workers=8)
!ls /content/fpc/data | wc -l

## 4. Convert to YOLO format

This copy's class names are wrong for doors and windows, so the script corrects them (see its header).

In [ ]:
open('/content/prepare_floorplancad.py', 'w', encoding='utf-8').write('"""Turn the Voxel51/FloorPlanCAD copy on Hugging Face into a YOLO detection dataset.\n\nThis copy\'s class NAMES are wrong for doors and windows (checked by eye on contact\nsheets of every class, Nov 2026); its boxes are fine. LABEL_MAP below uses what the\nboxes really mark:\n\n    name in the copy   what it marks     evidence\n    double_door        single door       one leaf, one quarter arc\n    sliding_door       double door       two leaves, two arcs\n    window             sliding door      thin strip, door codes (PM...)\n    stair              window            thin strip, window codes (ZC1805, C10524)\n\nImages are normalised to dark lines on white, because the plans the model will read\nare rendered that way from CAD (colour must not become a feature). Tiles from one\ndrawing (file name "<drawing>-<tile>.png") never appear in both train and val, so\nvalidation scores aren\'t inflated by near-duplicates.\n\nusage: python prepare_floorplancad.py --root <dataset folder> --out <yolo folder>\n"""\nimport argparse\nimport json\nimport os\nimport random\n\nimport cv2\nimport numpy as np\n\nLABEL_MAP = {"double_door": "door_single", "sliding_door": "door_double",\n             "window": "door_sliding", "stair": "window"}\nCLASSES = ["door_single", "door_double", "door_sliding", "window"]\nVAL_SHARE = 0.10\nSEED = 7\n\n\ndef clean_image(src, dst):\n    """RGBA drawing -> dark lines on white; any coloured line becomes dark."""\n    img = cv2.imread(src, cv2.IMREAD_UNCHANGED)\n    if img is None:\n        return False\n    if img.ndim == 2:\n        gray = img\n    else:\n        if img.shape[2] == 4:\n            alpha = img[..., 3:] / 255.0\n            img = (img[..., :3] * alpha + 255.0 * (1.0 - alpha)).astype(np.uint8)\n        gray = img.min(axis=2)          # min, not luma: a yellow line must not fade out\n    ink = np.clip((255.0 - gray) * 3.0, 0, 255)\n    cv2.imwrite(dst, (255.0 - ink).astype(np.uint8), [cv2.IMWRITE_JPEG_QUALITY, 92])\n    return True\n\n\ndef yolo_lines(detections):\n    """FiftyOne boxes (top-left x, y, w, h, normalised) -> YOLO label lines (centre x, y, w, h)."""\n    out = []\n    for d in detections:\n        name = LABEL_MAP.get(d["label"])\n        if name is None:\n            continue\n        x, y, w, h = d["bounding_box"]\n        x0, y0 = max(0.0, x), max(0.0, y)\n        x1, y1 = min(1.0, x + w), min(1.0, y + h)\n        if x1 - x0 < 0.002 or y1 - y0 < 0.002:\n            continue\n        out.append(f"{CLASSES.index(name)} {(x0 + x1) / 2:.6f} {(y0 + y1) / 2:.6f} {x1 - x0:.6f} {y1 - y0:.6f}")\n    return out\n\n\ndef split_by_drawing(names):\n    """{file name: "train"|"val"}, whole drawings at a time."""\n    drawings = sorted({n.split("-")[0] for n in names})\n    random.Random(SEED).shuffle(drawings)\n    val = set(drawings[:max(1, round(len(drawings) * VAL_SHARE))])\n    return {n: ("val" if n.split("-")[0] in val else "train") for n in names}\n\n\ndef prepare(root, out, images_dir="data"):\n    with open(os.path.join(root, "samples.json"), encoding="utf-8") as f:\n        samples = json.load(f)["samples"]\n    present = []\n    for s in samples:\n        name = s["filepath"].split("/")[-1]\n        if os.path.isfile(os.path.join(root, images_dir, name)):\n            present.append((name, s))\n    split = split_by_drawing([n for n, _ in present])\n    for part in ("train", "val"):\n        os.makedirs(os.path.join(out, "images", part), exist_ok=True)\n        os.makedirs(os.path.join(out, "labels", part), exist_ok=True)\n    stats = {"train": [0, 0], "val": [0, 0]}\n    counts = dict.fromkeys(CLASSES, 0)\n    for name, s in present:\n        part = split[name]\n        stem = os.path.splitext(name)[0]\n        if not clean_image(os.path.join(root, images_dir, name), os.path.join(out, "images", part, stem + ".jpg")):\n            continue\n        lines = yolo_lines((s.get("ground_truth") or {}).get("detections", []))\n        with open(os.path.join(out, "labels", part, stem + ".txt"), "w", encoding="utf-8") as f:\n            f.write("\\n".join(lines))\n        stats[part][0] += 1\n        stats[part][1] += len(lines)\n        for line in lines:\n            counts[CLASSES[int(line.split()[0])]] += 1\n    with open(os.path.join(out, "data.yaml"), "w", encoding="utf-8") as f:\n        f.write(f"path: {os.path.abspath(out)}\\ntrain: images/train\\nval: images/val\\nnames:\\n")\n        for i, c in enumerate(CLASSES):\n            f.write(f"  {i}: {c}\\n")\n    return {"images_boxes": stats, "classes": counts}\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--root", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--images", default="data")\n    a = ap.parse_args()\n    print(prepare(a.root, a.out, a.images))\n')

In [ ]:
!python /content/prepare_floorplancad.py --root /content/fpc --out /content/yolo

## 5. Train (2-4 hours; re-run resumes if the session ends)

In [ ]:
open('/content/train_yolo.py', 'w', encoding='utf-8').write('"""Train the door/window detector. Starts from official YOLOv8n weights; if a run already\nexists in the project folder (a Colab session ends, a laptop sleeps) it resumes it.\n\nusage: python train_yolo.py --data <data.yaml> --project <folder> [--epochs 60] [--imgsz 800]\n                            [--batch 16] [--device auto|mps|0|cpu]\n"""\nimport argparse\nimport os\n\nimport torch\nfrom ultralytics import YOLO\n\nRUN = "door_window"\n# Plans are line drawings with no "right way up" and no colour.\nAUGMENT = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, flipud=0.5, fliplr=0.5,\n               degrees=0.0, shear=0.0, perspective=0.0, scale=0.3, translate=0.1, mosaic=0.5)\n\n\ndef pick_device(device="auto"):\n    if device != "auto":\n        return device\n    if torch.cuda.is_available():\n        return "0"       # first NVIDIA GPU, in YOLO\'s device naming\n    if torch.backends.mps.is_available():\n        return "mps"\n    return "cpu"\n\n\ndef _unfinished(last):\n    """A run in progress saves its optimizer and epoch. A finished run\'s last.pt is stripped of\n    them, and resuming from it silently falls back to YOLO\'s default demo dataset."""\n    ckpt = torch.load(last, map_location="cpu", weights_only=False)   # our own file, from this run\n    return ckpt.get("optimizer") is not None and ckpt.get("epoch", -1) >= 0\n\n\ndef train(data, project, epochs=60, imgsz=800, batch=16, patience=15, device="auto"):\n    device = pick_device(device)\n    last = os.path.join(project, RUN, "weights", "last.pt")\n    best = os.path.join(project, RUN, "weights", "best.pt")\n    if os.path.isfile(last):\n        if not _unfinished(last):\n            print("This run already finished; nothing to do. Delete the run folder to train again.")\n            return best\n        YOLO(last).train(resume=True, device=device, batch=batch)\n        return best\n    YOLO("yolov8n.pt").train(\n        data=data, project=project, name=RUN, exist_ok=True, seed=7, device=device,\n        epochs=epochs, imgsz=imgsz, batch=batch, patience=patience, **AUGMENT,\n    )\n    return best\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--project", required=True)\n    ap.add_argument("--epochs", type=int, default=60)\n    ap.add_argument("--imgsz", type=int, default=800)\n    ap.add_argument("--batch", type=int, default=16)\n    ap.add_argument("--device", default="auto")\n    a = ap.parse_args()\n    print("best weights:", train(a.data, a.project, a.epochs, a.imgsz, a.batch, device=a.device))\n')

In [ ]:
!cd /content && python /content/train_yolo.py --data /content/yolo/data.yaml --project "$PROJECT" --epochs 60 --imgsz 800 --batch 16

## 6. Final scores, and save the model to your Drive

In [ ]:
import shutil
from ultralytics import YOLO
best = f'{PROJECT}/door_window/weights/best.pt'
metrics = YOLO(best).val(data='/content/yolo/data.yaml', imgsz=800, split='val')
for i, name in metrics.names.items():
    p, r, map50, map5095 = metrics.class_result(metrics.ap_class_index.tolist().index(i))
    print(f'{name:<14} precision {p:.3f}  recall {r:.3f}  mAP50 {map50:.3f}')
shutil.copy(best, f'{PROJECT}/dreamspace_door_window.pt')
print('Saved: Google Drive > DreamSpace_training > dreamspace_door_window.pt  (download this file)')